# Benchmark OCR
Compara PP-OCRv6 y PP-OCRv5 con etiquetas de texto conocidas.


In [ ]:
!pip install -q "paddleocr>=3.0.0" "paddlepaddle>=3.0.0" pandas jiwer


In [ ]:
from pathlib import Path
import time

import pandas as pd
from jiwer import cer
from paddleocr import PaddleOCR

DATA_ROOT = Path("/kaggle/input/tu-dataset-ocr")
LABELS_CSV = DATA_ROOT / "labels.csv"
labels = pd.read_csv(LABELS_CSV)
labels.head()


In [ ]:
def extract_text(predictions):
    texts = []
    for prediction in predictions:
        payload = prediction.json
        data = payload.get("res", payload)
        texts.extend([str(value) for value in data.get("rec_texts", []) if str(value).strip()])
    return " ".join(texts)

configs = {
    "ppocrv6": {},
    "ppocrv5": {"ocr_version": "PP-OCRv5"},
}
rows = []
for name, config in configs.items():
    pipeline = PaddleOCR(use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False, lang="es", **config)
    truth, pred = [], []
    start = time.perf_counter()
    for row in labels.itertuples(index=False):
        expected = str(row.text)
        predicted = extract_text(pipeline.predict(str(DATA_ROOT / row.image)))
        truth.append(expected)
        pred.append(predicted)
    elapsed = time.perf_counter() - start
    rows.append({
        "model": name,
        "cer": cer(truth, pred),
        "exact_match": sum(a == b for a, b in zip(truth, pred)) / len(truth),
        "seconds": elapsed,
    })

results = pd.DataFrame(rows).sort_values(["cer", "seconds"]).reset_index(drop=True)
results


In [ ]:
results.to_csv("/kaggle/working/ocr_benchmark.csv", index=False)
